# Module 06: Feature Engineering

**CareerAlign GCP Professional ML Engineer**

This notebook covers:
1. Feature Store: create entity types, add features, ingest data (SDK code)
2. Numerical feature engineering: scaling, bucketizing
3. Categorical encoding: one-hot, label encoding, hashing trick
4. Feature crosses with pandas
5. tf.Transform preprocessing function (local)
6. Feature importance with sklearn
7. Model comparison: with vs without feature engineering

> **Note**: Cells marked with `# REQUIRES GCP` need a GCP project. All other cells run locally.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rajvermatx/careeralign.com/blob/main/gcp-mle/notebooks/06-feature-engineering.ipynb)

---
## Setup & Installations

In [ ]:
# Install required packages
!pip install -q pandas numpy scikit-learn matplotlib

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import (
    StandardScaler, MinMaxScaler, LabelEncoder,
    OneHotEncoder, KBinsDiscretizer
)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report
from sklearn.feature_selection import mutual_info_classif
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
print("Setup complete!")

---
## 1. Vertex AI Feature Store (SDK Reference)

This section shows the Feature Store SDK code. **Requires GCP** to run.

The Feature Store provides:
- **Entity types**: logical groupings (e.g., "users", "products")
- **Features**: individual measurable properties within entity types
- **Online serving**: low-latency lookups for real-time prediction
- **Offline serving**: batch export with point-in-time joins for training

In [ ]:
# REQUIRES GCP - Feature Store SDK code (reference only)
# Uncomment and run in a GCP environment with Vertex AI enabled

feature_store_code = """
from google.cloud import aiplatform

# Initialize
aiplatform.init(project="my-project", location="us-central1")

# === Step 1: Create Featurestore ===
fs = aiplatform.Featurestore.create(
    featurestore_id="ecommerce_features",
    online_store_fixed_node_count=1  # for online serving
)

# === Step 2: Create Entity Type ===
user_entity = fs.create_entity_type(
    entity_type_id="users",
    description="User-level features for churn and recommendation models"
)

# === Step 3: Add Features ===
user_entity.batch_create_features(
    feature_configs={
        "age": {"value_type": "INT64", "description": "User age in years"},
        "total_purchases": {"value_type": "DOUBLE", "description": "Lifetime purchase total"},
        "account_age_days": {"value_type": "INT64", "description": "Days since signup"},
        "region": {"value_type": "STRING", "description": "Geographic region"},
        "is_premium": {"value_type": "BOOL", "description": "Premium membership status"},
        "avg_session_minutes": {"value_type": "DOUBLE", "description": "Average session duration"},
    }
)

# === Step 4: Ingest Features from BigQuery ===
user_entity.ingest_from_bq(
    feature_ids=["age", "total_purchases", "account_age_days", "region", "is_premium"],
    feature_time="feature_timestamp",  # column with timestamps
    bq_source_uri="bq://project.dataset.user_features_table",
    entity_id_field="user_id"
)

# === Step 5: Online Serving (real-time lookup) ===
online_features = fs.read_feature_values(
    entity_type_id="users",
    entity_ids=["user_001", "user_002"],
    feature_ids=["age", "total_purchases", "region"]
)

# === Step 6: Offline Serving (batch for training) ===
# Point-in-time join with entity IDs + timestamps
training_data = fs.batch_serve_to_bq(
    bq_destination_output_uri="bq://project.dataset.training_data",
    serving_feature_ids={
        "users": ["age", "total_purchases", "account_age_days", "region"]
    },
    read_instances_uri="bq://project.dataset.entity_timestamps"  # entity_id + timestamp pairs
)
"""

print("Feature Store SDK Reference Code:")
print("=" * 50)
print(feature_store_code)
print("\nNote: This code requires a GCP project with Vertex AI Feature Store enabled.")

---
## 2. Numerical Feature Engineering

Techniques: scaling (min-max, z-score), log transform, and bucketizing.

In [ ]:
# --- Create a dataset for feature engineering demos ---

n = 2000
np.random.seed(42)

# Numerical features with different distributions
age = np.random.normal(40, 12, n).clip(18, 80).astype(int)
income = np.random.lognormal(10.5, 0.8, n).round(2)  # right-skewed
session_minutes = np.random.exponential(15, n).round(1)  # exponential
purchases = np.random.poisson(5, n)  # count data

# Categorical features
regions = np.random.choice(['US-East', 'US-West', 'EU', 'APAC', 'LATAM'], n, p=[0.3, 0.25, 0.2, 0.15, 0.1])
devices = np.random.choice(['mobile', 'desktop', 'tablet'], n, p=[0.5, 0.35, 0.15])
membership = np.random.choice(['free', 'basic', 'premium', 'enterprise'], n, p=[0.5, 0.25, 0.15, 0.10])

# Timestamps
hours = np.random.randint(0, 24, n)
days_of_week = np.random.randint(0, 7, n)

# Label (binary, imbalanced)
# Make label somewhat dependent on features
logit = -2 + 0.02 * age + 0.3 * np.log1p(purchases) + 0.5 * (np.array(membership) == 'premium').astype(float)
prob = 1 / (1 + np.exp(-logit))
labels = (np.random.random(n) < prob).astype(int)

df = pd.DataFrame({
    'age': age, 'income': income, 'session_minutes': session_minutes,
    'purchases': purchases, 'region': regions, 'device': devices,
    'membership': membership, 'hour_of_day': hours,
    'day_of_week': days_of_week, 'label': labels
})

print(f"Dataset: {n} rows, {len(df.columns)} columns")
print(f"Label distribution: {dict(df['label'].value_counts())}")
print(f"\nNumerical feature statistics:")
df[['age', 'income', 'session_minutes', 'purchases']].describe().round(2)

In [ ]:
# --- Scaling techniques ---

# Split first to avoid leakage
df_train, df_test = train_test_split(df, test_size=0.2, random_state=42, stratify=df['label'])
print(f"Train: {len(df_train)}, Test: {len(df_test)}")

numerical_cols = ['age', 'income', 'session_minutes', 'purchases']

# 1. Min-Max Scaling [0, 1]
minmax = MinMaxScaler()
minmax.fit(df_train[numerical_cols])  # Fit on train only!
df_train_mm = pd.DataFrame(
    minmax.transform(df_train[numerical_cols]),
    columns=[f"{c}_minmax" for c in numerical_cols],
    index=df_train.index
)

# 2. Z-Score Standardization
zscore = StandardScaler()
zscore.fit(df_train[numerical_cols])
df_train_zs = pd.DataFrame(
    zscore.transform(df_train[numerical_cols]),
    columns=[f"{c}_zscore" for c in numerical_cols],
    index=df_train.index
)

# 3. Log Transform (for skewed features)
df_train_log = pd.DataFrame({
    'income_log': np.log1p(df_train['income']),
    'session_minutes_log': np.log1p(df_train['session_minutes'])
}, index=df_train.index)

# 4. Bucketizing
binner = KBinsDiscretizer(n_bins=5, encode='ordinal', strategy='quantile')
binner.fit(df_train[['age']])
df_train_bucket = pd.DataFrame({
    'age_bucket': binner.transform(df_train[['age']]).ravel().astype(int)
}, index=df_train.index)

# Display comparison
comparison = pd.concat([
    df_train[['age', 'income']].head(8).reset_index(drop=True),
    df_train_mm[['age_minmax', 'income_minmax']].head(8).reset_index(drop=True),
    df_train_zs[['age_zscore', 'income_zscore']].head(8).reset_index(drop=True),
    df_train_log[['income_log']].head(8).reset_index(drop=True),
    df_train_bucket.head(8).reset_index(drop=True)
], axis=1)

print("\nScaling Comparison (first 8 rows):")
comparison.round(3)

---
## 3. Categorical Encoding

In [ ]:
# --- Categorical encoding techniques ---

cat_cols = ['region', 'device', 'membership']

# 1. One-Hot Encoding
ohe = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
ohe.fit(df_train[cat_cols])
ohe_features = ohe.transform(df_train[cat_cols])
ohe_names = ohe.get_feature_names_out(cat_cols)

print("=== One-Hot Encoding ===")
print(f"Input: {len(cat_cols)} categorical columns")
print(f"Output: {ohe_features.shape[1]} binary columns")
print(f"Feature names: {list(ohe_names)}")
print(f"\nFirst 3 rows:")
print(pd.DataFrame(ohe_features[:3], columns=ohe_names).to_string())
print()

In [ ]:
# 2. Label Encoding (ordinal)
le_dict = {}
le_features = pd.DataFrame(index=df_train.index)
for col in cat_cols:
    le = LabelEncoder()
    le.fit(df_train[col])
    le_features[f"{col}_le"] = le.transform(df_train[col])
    le_dict[col] = le

print("=== Label Encoding ===")
for col in cat_cols:
    mapping = dict(zip(le_dict[col].classes_, le_dict[col].transform(le_dict[col].classes_)))
    print(f"  {col}: {mapping}")

print(f"\nFirst 5 rows:")
pd.concat([df_train[cat_cols].head().reset_index(drop=True),
           le_features.head().reset_index(drop=True)], axis=1)

In [ ]:
# 3. Hashing Trick
# Simulates ML.HASH_BUCKETS in BQML

def hash_encode(series, num_buckets):
    """Hash strings into fixed number of buckets."""
    return series.apply(lambda x: hash(str(x)) % num_buckets)

NUM_BUCKETS = 8

hash_features = pd.DataFrame(index=df_train.index)
for col in cat_cols:
    hash_features[f"{col}_hash{NUM_BUCKETS}"] = hash_encode(df_train[col], NUM_BUCKETS)

print(f"=== Feature Hashing (num_buckets={NUM_BUCKETS}) ===")
print("Advantage: Fixed output size regardless of cardinality")
print("Tradeoff: Collisions possible (different values -> same bucket)")
print()

# Show collisions
for col in cat_cols:
    unique_vals = df_train[col].nunique()
    unique_hashes = hash_features[f"{col}_hash{NUM_BUCKETS}"].nunique()
    collisions = unique_vals - unique_hashes
    print(f"  {col}: {unique_vals} categories -> {unique_hashes} buckets ({collisions} collisions)")

print(f"\nFirst 5 rows:")
pd.concat([df_train[cat_cols].head().reset_index(drop=True),
           hash_features.head().reset_index(drop=True)], axis=1)

---
## 4. Feature Crosses with Pandas

Feature crosses allow linear models to capture non-linear interactions.

In [ ]:
# --- Feature crosses ---

# Cross 1: region x device (categorical x categorical)
df_train_crosses = df_train.copy()
df_train_crosses['region_device'] = df_train['region'] + '_' + df_train['device']

# Cross 2: age_bucket x membership (numerical bucket x categorical)
df_train_crosses['age_bucket'] = pd.cut(
    df_train['age'], bins=[0, 25, 35, 50, 65, 100],
    labels=['18-25', '26-35', '36-50', '51-65', '65+']
).astype(str)
df_train_crosses['age_membership'] = df_train_crosses['age_bucket'] + '_' + df_train['membership']

# Cross 3: hour_bucket x day_type (time-based cross)
df_train_crosses['hour_bucket'] = pd.cut(
    df_train['hour_of_day'], bins=[-1, 6, 12, 18, 24],
    labels=['night', 'morning', 'afternoon', 'evening']
).astype(str)
df_train_crosses['day_type'] = np.where(df_train['day_of_week'] >= 5, 'weekend', 'weekday')
df_train_crosses['time_day_cross'] = df_train_crosses['hour_bucket'] + '_' + df_train_crosses['day_type']

print("=== Feature Crosses ===")
print(f"\n1. region x device ({df_train_crosses['region_device'].nunique()} unique values):")
print(f"   Examples: {df_train_crosses['region_device'].unique()[:5].tolist()}")

print(f"\n2. age_bucket x membership ({df_train_crosses['age_membership'].nunique()} unique values):")
print(f"   Examples: {df_train_crosses['age_membership'].unique()[:5].tolist()}")

print(f"\n3. time x day_type ({df_train_crosses['time_day_cross'].nunique()} unique values):")
print(f"   All values: {sorted(df_train_crosses['time_day_cross'].unique().tolist())}")

# Show the cardinality explosion
print(f"\nCardinality analysis:")
print(f"  region alone: {df_train['region'].nunique()} values")
print(f"  device alone: {df_train['device'].nunique()} values")
print(f"  region x device cross: {df_train_crosses['region_device'].nunique()} values")
print(f"  (Theoretical max: {df_train['region'].nunique()} * {df_train['device'].nunique()} = {df_train['region'].nunique() * df_train['device'].nunique()})")

In [ ]:
# --- XOR demonstration: why crosses matter for linear models ---

# Create XOR-like data
np.random.seed(42)
n_xor = 500
x1 = np.random.choice([0, 1], n_xor)
x2 = np.random.choice([0, 1], n_xor)
# XOR: label = 1 when x1 != x2
y_xor = (x1 != x2).astype(int)
# Add cross feature
x1_x2_cross = x1 * x2

# Model without cross
X_no_cross = np.column_stack([x1, x2])
lr_no_cross = LogisticRegression(random_state=42)
lr_no_cross.fit(X_no_cross, y_xor)
acc_no_cross = accuracy_score(y_xor, lr_no_cross.predict(X_no_cross))

# Model with cross
X_with_cross = np.column_stack([x1, x2, x1_x2_cross])
lr_with_cross = LogisticRegression(random_state=42)
lr_with_cross.fit(X_with_cross, y_xor)
acc_with_cross = accuracy_score(y_xor, lr_with_cross.predict(X_with_cross))

print("=== XOR Problem: Feature Cross Demo ===")
print(f"Without cross (x1, x2):        Accuracy = {acc_no_cross:.4f}")
print(f"With cross (x1, x2, x1*x2):    Accuracy = {acc_with_cross:.4f}")
print()
print("The feature cross x1*x2 makes XOR linearly separable!")
print("This is the fundamental insight behind feature crosses.")
print("\nBQML equivalent: ML.FEATURE_CROSS(STRUCT(x1, x2))")

---
## 5. tf.Transform Preprocessing Function (Local Example)

This shows the tf.Transform pattern conceptually. For full execution, install `tensorflow-transform`.

In [ ]:
# --- tf.Transform preprocessing_fn (reference code) ---

tft_code = """
import tensorflow_transform as tft

def preprocessing_fn(inputs):
    \"\"\"Define feature transformations for tf.Transform.
    
    Analyzers (full-pass): tft.mean, tft.min, tft.max, tft.vocabulary
        - Run once over the full dataset during analysis phase
        - Results are baked into the saved TF graph as constants
    
    Mappers (per-example): tft.scale_to_0_1, tft.apply_vocabulary
        - Applied to each example at both training and serving time
        - Use constants computed by analyzers
    \"\"\"    
    outputs = {}
    
    # === Numerical Features ===
    
    # Scale age to [0, 1] using dataset min/max (analyzer)
    outputs['age_scaled'] = tft.scale_to_0_1(inputs['age'])
    
    # Z-score normalize income (analyzer computes mean, std)
    outputs['income_normalized'] = tft.scale_to_z_score(inputs['income'])
    
    # Bucketize into quantile-based bins (analyzer computes quantiles)
    outputs['age_bucket'] = tft.bucketize(inputs['age'], num_buckets=5)
    outputs['income_bucket'] = tft.bucketize(inputs['income'], num_buckets=10)
    
    # Log transform (per-example, no analyzer needed)
    import tensorflow as tf
    outputs['income_log'] = tf.math.log1p(tf.cast(inputs['income'], tf.float32))
    
    # === Categorical Features ===
    
    # Compute vocabulary and map to integer indices (analyzer)
    outputs['region_index'] = tft.compute_and_apply_vocabulary(
        inputs['region'], top_k=100
    )
    
    # Hash to fixed buckets (per-example, no analyzer)
    outputs['device_hash'] = tft.hash_strings(
        inputs['device'], hash_buckets=10
    )
    
    # === Text Features ===
    
    # Tokenize, build vocab, compute TF-IDF (analyzer)
    tokens = tft.compute_and_apply_vocabulary(inputs['description'])
    outputs['description_tfidf'] = tft.tfidf(tokens, vocab_size=5000)
    
    # === Temporal Features ===
    
    # Cyclical encoding (per-example, no analyzer)
    import math
    hour = tf.cast(inputs['hour_of_day'], tf.float32)
    outputs['hour_sin'] = tf.sin(2 * math.pi * hour / 24.0)
    outputs['hour_cos'] = tf.cos(2 * math.pi * hour / 24.0)
    
    # Pass through label
    outputs['label'] = inputs['label']
    
    return outputs
"""

print("tf.Transform preprocessing_fn Reference:")
print("=" * 50)
print(tft_code)
print("\nKey points:")
print("  1. Analyzers run ONCE over full data (compute stats)")
print("  2. Mappers run PER-EXAMPLE (apply transforms)")
print("  3. Exported as TF graph -> same transforms at serving time")
print("  4. Runs on Dataflow for scale")

In [ ]:
# --- Simulate tf.Transform locally with sklearn/numpy ---

# Cyclical encoding for temporal features
df_temporal = df_train[['hour_of_day', 'day_of_week']].copy()

# Hour cyclical encoding
df_temporal['hour_sin'] = np.sin(2 * np.pi * df_temporal['hour_of_day'] / 24)
df_temporal['hour_cos'] = np.cos(2 * np.pi * df_temporal['hour_of_day'] / 24)

# Day of week cyclical encoding
df_temporal['dow_sin'] = np.sin(2 * np.pi * df_temporal['day_of_week'] / 7)
df_temporal['dow_cos'] = np.cos(2 * np.pi * df_temporal['day_of_week'] / 7)

print("=== Cyclical Encoding Demo ===")
print("\nHour of Day -> sin/cos:")
sample_hours = pd.DataFrame({
    'hour': [0, 6, 12, 18, 23],
    'sin': np.sin(2 * np.pi * np.array([0, 6, 12, 18, 23]) / 24).round(4),
    'cos': np.cos(2 * np.pi * np.array([0, 6, 12, 18, 23]) / 24).round(4)
})
print(sample_hours.to_string(index=False))

print("\nNotice: hour 23 and hour 0 are adjacent in sin/cos space!")
h23_vec = np.array([np.sin(2*np.pi*23/24), np.cos(2*np.pi*23/24)])
h0_vec = np.array([np.sin(2*np.pi*0/24), np.cos(2*np.pi*0/24)])
h12_vec = np.array([np.sin(2*np.pi*12/24), np.cos(2*np.pi*12/24)])
print(f"  Distance(hour 23, hour 0)  = {np.linalg.norm(h23_vec - h0_vec):.4f}")
print(f"  Distance(hour 23, hour 12) = {np.linalg.norm(h23_vec - h12_vec):.4f}")

---
## 6. Feature Importance with sklearn

In [ ]:
# --- Build full feature set for importance analysis ---

# Prepare all features
feature_df = df_train.copy()

# Encode categoricals
for col in cat_cols:
    feature_df[f"{col}_enc"] = le_dict[col].transform(feature_df[col])

# Add engineered features
feature_df['income_log'] = np.log1p(feature_df['income'])
feature_df['session_log'] = np.log1p(feature_df['session_minutes'])
feature_df['hour_sin'] = np.sin(2 * np.pi * feature_df['hour_of_day'] / 24)
feature_df['hour_cos'] = np.cos(2 * np.pi * feature_df['hour_of_day'] / 24)
feature_df['dow_sin'] = np.sin(2 * np.pi * feature_df['day_of_week'] / 7)
feature_df['dow_cos'] = np.cos(2 * np.pi * feature_df['day_of_week'] / 7)
feature_df['is_weekend'] = (feature_df['day_of_week'] >= 5).astype(int)
feature_df['purchase_income_ratio'] = feature_df['purchases'] / (feature_df['income'] / 1000)

feature_cols = [
    'age', 'income_log', 'session_log', 'purchases',
    'region_enc', 'device_enc', 'membership_enc',
    'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos',
    'is_weekend', 'purchase_income_ratio'
]

X_train = feature_df[feature_cols].values
y_train_labels = feature_df['label'].values

# --- Random Forest Feature Importance ---
rf = RandomForestClassifier(n_estimators=200, max_depth=10, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train_labels)

importance_rf = pd.Series(rf.feature_importances_, index=feature_cols).sort_values(ascending=False)

print("=== Random Forest Feature Importance ===")
for feat, imp in importance_rf.items():
    bar = '#' * int(imp * 200)
    print(f"  {feat:<25s} {imp:.4f} {bar}")

In [ ]:
# --- Mutual Information (filter method) ---

mi_scores = mutual_info_classif(X_train, y_train_labels, random_state=42)
importance_mi = pd.Series(mi_scores, index=feature_cols).sort_values(ascending=False)

print("=== Mutual Information Scores ===")
for feat, mi in importance_mi.items():
    bar = '#' * int(mi * 500)
    print(f"  {feat:<25s} {mi:.4f} {bar}")

print("\n=== Correlation Matrix (numerical features) ===")
corr_cols = ['age', 'income_log', 'session_log', 'purchases', 'purchase_income_ratio']
corr_matrix = feature_df[corr_cols].corr().round(3)
print(corr_matrix.to_string())

# Check for highly correlated pairs (> 0.8)
print("\nHighly correlated pairs (|r| > 0.8):")
found = False
for i in range(len(corr_cols)):
    for j in range(i+1, len(corr_cols)):
        r = abs(corr_matrix.iloc[i, j])
        if r > 0.8:
            print(f"  {corr_cols[i]} <-> {corr_cols[j]}: r = {corr_matrix.iloc[i, j]:.3f}")
            found = True
if not found:
    print("  None found (all |r| < 0.8)")

---
## 7. Model Comparison: With vs Without Feature Engineering

In [ ]:
# --- Prepare test set identically ---

test_feature_df = df_test.copy()
for col in cat_cols:
    test_feature_df[f"{col}_enc"] = le_dict[col].transform(test_feature_df[col])
test_feature_df['income_log'] = np.log1p(test_feature_df['income'])
test_feature_df['session_log'] = np.log1p(test_feature_df['session_minutes'])
test_feature_df['hour_sin'] = np.sin(2 * np.pi * test_feature_df['hour_of_day'] / 24)
test_feature_df['hour_cos'] = np.cos(2 * np.pi * test_feature_df['hour_of_day'] / 24)
test_feature_df['dow_sin'] = np.sin(2 * np.pi * test_feature_df['day_of_week'] / 7)
test_feature_df['dow_cos'] = np.cos(2 * np.pi * test_feature_df['day_of_week'] / 7)
test_feature_df['is_weekend'] = (test_feature_df['day_of_week'] >= 5).astype(int)
test_feature_df['purchase_income_ratio'] = test_feature_df['purchases'] / (test_feature_df['income'] / 1000)

X_test_eng = test_feature_df[feature_cols].values
y_test_labels = test_feature_df['label'].values

print(f"Test set: {len(X_test_eng)} samples")
print(f"Engineered features: {len(feature_cols)}")

In [ ]:
# --- Model 1: Raw features only ---

raw_cols = ['age', 'income', 'session_minutes', 'purchases']
# Encode cats simply
for col in cat_cols:
    raw_cols.append(f"{col}_enc")

X_train_raw = feature_df[raw_cols].values
X_test_raw = test_feature_df[raw_cols].values

scaler = StandardScaler()
X_train_raw_s = scaler.fit_transform(X_train_raw)
X_test_raw_s = scaler.transform(X_test_raw)

# Logistic Regression with raw features
lr_raw = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)
lr_raw.fit(X_train_raw_s, y_train_labels)
y_pred_raw = lr_raw.predict(X_test_raw_s)
y_proba_raw = lr_raw.predict_proba(X_test_raw_s)[:, 1]

print("=== Model 1: Logistic Regression with RAW Features ===")
print(f"Features ({len(raw_cols)}): {raw_cols}")
print(f"Accuracy: {accuracy_score(y_test_labels, y_pred_raw):.4f}")
print(f"ROC AUC:  {roc_auc_score(y_test_labels, y_proba_raw):.4f}")

In [ ]:
# --- Model 2: Engineered features ---

scaler_eng = StandardScaler()
X_train_eng_s = scaler_eng.fit_transform(X_train)
X_test_eng_s = scaler_eng.transform(X_test_eng)

lr_eng = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)
lr_eng.fit(X_train_eng_s, y_train_labels)
y_pred_eng = lr_eng.predict(X_test_eng_s)
y_proba_eng = lr_eng.predict_proba(X_test_eng_s)[:, 1]

print("=== Model 2: Logistic Regression with ENGINEERED Features ===")
print(f"Features ({len(feature_cols)}): {feature_cols}")
print(f"Accuracy: {accuracy_score(y_test_labels, y_pred_eng):.4f}")
print(f"ROC AUC:  {roc_auc_score(y_test_labels, y_proba_eng):.4f}")

In [ ]:
# --- Model 3: Gradient Boosting with engineered features ---

gb = GradientBoostingClassifier(
    n_estimators=200, max_depth=5, learning_rate=0.1, random_state=42
)
gb.fit(X_train, y_train_labels)
y_pred_gb = gb.predict(X_test_eng)
y_proba_gb = gb.predict_proba(X_test_eng)[:, 1]

print("=== Model 3: Gradient Boosting with ENGINEERED Features ===")
print(f"Accuracy: {accuracy_score(y_test_labels, y_pred_gb):.4f}")
print(f"ROC AUC:  {roc_auc_score(y_test_labels, y_proba_gb):.4f}")

In [ ]:
# --- Final Comparison ---

print("=" * 65)
print("  Feature Engineering Impact: Model Comparison")
print("=" * 65)
print(f"{'Model':<40} {'Accuracy':>10} {'ROC AUC':>10}")
print("-" * 65)
print(f"{'LR + Raw Features (' + str(len(raw_cols)) + ' features)':<40} {accuracy_score(y_test_labels, y_pred_raw):>10.4f} {roc_auc_score(y_test_labels, y_proba_raw):>10.4f}")
print(f"{'LR + Engineered Features (' + str(len(feature_cols)) + ' features)':<40} {accuracy_score(y_test_labels, y_pred_eng):>10.4f} {roc_auc_score(y_test_labels, y_proba_eng):>10.4f}")
print(f"{'GB + Engineered Features (' + str(len(feature_cols)) + ' features)':<40} {accuracy_score(y_test_labels, y_pred_gb):>10.4f} {roc_auc_score(y_test_labels, y_proba_gb):>10.4f}")
print("=" * 65)
print()
print("Key takeaways:")
print("  1. Feature engineering improves linear model performance")
print("  2. Log transforms help with skewed distributions")
print("  3. Cyclical encoding captures temporal patterns")
print("  4. Tree-based models benefit less from manual transforms")
print("     (they learn non-linear patterns through splits)")

---
## Summary

| Topic | Key Takeaway |
|-------|-------------|
| **Feature Store** | Centralized, versioned, online+offline serving. Use for shared features across teams |
| **Numerical** | Min-max for bounded, z-score for Gaussian, log for skewed, bucketize for non-linear |
| **Categorical** | One-hot for low cardinality, hashing for high cardinality, embeddings for very high |
| **Feature Crosses** | Enable linear models to learn interactions. Combine with hashing for scale |
| **tf.Transform** | Analyze-and-transform pattern. Saved TF graph prevents training-serving skew |
| **Feature Selection** | RF importance, mutual information, correlation. Use embedded methods (L1) for automatic selection |
| **Impact** | Feature engineering typically improves AUC, especially for linear models |